# 1. Import das bibliotecas necessárias

In [ ]:
%pip install scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import joblib


# 2. Carregamento dos dados

## Sobre o dataset

### Contexto
Uma empresa fictícia de telecomunicações que forneceu serviços de telefonia fixa e internet para 7.043 clientes na Califórnia no terceiro trimestre.

### Esquema de variáveis (conforme o kaggle)

- ID do cliente : Um ID exclusivo que identifica cada cliente.

- Contagem : Um valor usado em relatórios/painéis para somar o número de clientes em um conjunto filtrado.

- País : O país de residência principal do cliente.

- Estado : O estado da residência principal do cliente.

- Cidade : A cidade da residência principal do cliente.

- CEP : O CEP da residência principal do cliente.

- Lat Long : A combinação da latitude e longitude da residência principal do cliente.

- Latitude : A latitude da residência principal do cliente.

- Longitude : A longitude da residência principal do cliente.

- Gênero : O gênero do cliente: Masculino, Feminino

- Idoso : Indica se o cliente tem 65 anos ou mais: Sim, Não

- Parceiro : Indique se o cliente tem um parceiro: Sim, Não

- Dependentes : Indica se o cliente mora com algum dependente: Sim, Não. Dependentes podem ser filhos, pais, avós, etc.

- Tempo de permanência no relacionamento : Indica o número total de meses que o cliente permaneceu com a empresa até o final do trimestre especificado acima.

- Serviço Telefônico : Indica se o cliente possui assinatura de serviço de telefonia fixa com a empresa: Sim, Não

- Linhas Múltiplas : Indica se o cliente possui várias linhas telefônicas com a empresa: Sim, Não

- Serviço de Internet : Indica se o cliente possui assinatura de serviço de Internet com a empresa: Não, DSL, Fibra Óptica, Cabo.

- Segurança online : Indica se o cliente subscreve um serviço adicional de segurança online fornecido pela empresa: Sim, Não

- Backup online : Indica se o cliente subscreve um serviço adicional de backup online fornecido pela empresa: Sim, Não

- Proteção do dispositivo : Indica se o cliente subscreveu um plano adicional de proteção para o seu equipamento de Internet fornecido pela empresa: Sim, Não

- Suporte técnico : Indica se o cliente subscreveu um plano adicional de suporte técnico da empresa com tempos de espera reduzidos: Sim, Não

- Streaming de TV : Indica se o cliente utiliza seu serviço de internet para assistir a programas de televisão de um provedor terceirizado: Sim, Não. A empresa não cobra nenhuma taxa adicional por este serviço.

- Streaming de filmes : Indica se o cliente utiliza seu serviço de internet para assistir a filmes de um provedor terceirizado: Sim, Não. A empresa não cobra nenhuma taxa adicional por este serviço.

- Contrato : Indica o tipo de contrato atual do cliente: Mensal, Anual ou Bienal.

- Fatura eletrônica : Indica se o cliente optou pela fatura eletrônica: Sim, Não

- Método de pagamento : Indica como o cliente pagará sua conta: Débito bancário, Cartão de crédito, Cheque enviado pelo correio

- Cobrança mensal : Indica o valor total cobrado mensalmente pelo cliente por todos os serviços prestados pela empresa.

- Total de Cobranças : Indica o total de cobranças do cliente, calculado até o final do trimestre especificado acima.

- Rótulo de Churn : Sim = o cliente deixou a empresa neste trimestre. Não = o cliente permaneceu com a empresa. Diretamente relacionado ao Valor do Churn.

- Valor de Churn : 1 = o cliente deixou a empresa neste trimestre. 0 = o cliente permaneceu com a empresa. Diretamente relacionado ao Rótulo de Churn.

- Churn Score : Um valor de 0 a 100 calculado usando a ferramenta preditiva IBM SPSS Modeler. O modelo incorpora múltiplos fatores conhecidos por causarem churn. Quanto maior a pontuação, maior a probabilidade de o cliente cancelar o serviço.

- CLTV : Valor do Tempo de Vida do Cliente. O CLTV previsto é calculado usando fórmulas corporativas e dados existentes. Quanto maior o valor, mais valioso o cliente. Clientes de alto valor devem ser monitorados quanto à possibilidade de cancelamento (churn).

- Motivo do Cancelamento : A razão específica pela qual um cliente deixou a empresa. Diretamente relacionado à Categoria de Cancelamento.

In [ ]:
# Carregando o dataset para o repositório

path = '../data/Telco_customer_churn.xlsx'

df = pd.read_excel(path)

In [ ]:
df.rename(columns={
            'CustomerID': 'customer_id',
            'Count': 'count',	
            'Country': 'country',	
            'State': 'state',
            'City': 'city',
            'Zip Code': 'zip_code',
            'Lat Long': 'lat_long',
            'Latitude': 'latitude',
            'Longitude': 'longitude',	
            'Gender': 'gender',
            'Senior Citizen': 'senior_citizen',
            'Partner': 'partner',	
            'Dependents': 'dependents',	
            'Tenure Months': 'tenure_months',	
            'Phone Service': 'phone_service',	
            'Multiple Lines': 'multiple_lines',	
            'Internet Service': 'internet_service',	
            'Online Security': 'online_security',	
            'Online Backup': 'online_backup',	
            'Device Protection': 'device_protection',	
            'Tech Support': 'tech_support',	
            'Streaming TV': 'streaming_tv',	
            'Streaming Movies': 'streaming_movies',	
            'Contract': 'contract',	
            'Paperless Billing': 'paperless_billing',	
            'Payment Method': 'payment_method',
            'Monthly Charges': 'monthly_charges',	
            'Total Charges': 'total_charges',	
            'Churn Label': 'churn_label',	
            'Churn Value': 'churn_value',	
            'Churn Score': 'churn_score',	
            'CLTV': 'cltv',	
            'Churn Reason': 'churn_reason'},
            inplace=True
)

In [ ]:

print(f"=========== Primeiras análises sobre o dataset  ===========")
print(f"         ")
print(f"Shape dos dados: {df.shape}")
print(f"         ")
print(f"Primeiras linhas do datase:")
display(df.head(10))
print(f"         ")
print(f"Informações dos dados: {df.info()}")
print(f"         ")
print(f"Informações estatísticas dos dados: {df.describe()}")

# Tratamento de missing values

In [ ]:
df.info()

In [ ]:
print("=== TRATAMENTO DE MISSING VALUES ===\n")

# Copiar o dataframe original para preservar os dados
df_clean = df.copy()

# Imputação para variáveis categóricas: moda
cat_missing = ['churn_reason']
for col in cat_missing:
    if col in df_clean.columns:
        mode = df_clean[col].mode()[0]
        df_clean[col].fillna(mode, inplace=True)
        print(f"{col}: imputado com moda ('{mode}')")

print("\n✓ Tratamento de missing values concluído!")
print(f"Shape após imputação: {df_clean.shape}")
print("Valores ausentes restantes por coluna:")
print(df_clean.isnull().sum())

# Tratamento de One hot encoding

In [ ]:
df_clean.info()

In [ ]:
#categorical_cols = columns=['country','state','city','lat_long','gender','senior_citizen','partner','dependents','phone_service','multiple_lines','internet_service','device_protection','online_security','online_backup']
categorical_cols = df_clean.select_dtypes(include='object').columns
df_encoded = pd.get_dummies(df_clean,columns=categorical_cols, drop_first=True)

print(f"Shape pós enconding: {df_encoded.shape}")
df_encoded.head()

# Experimentação e MVP (regressão logística - baseline)

In [ ]:
from sklearn.model_selection import train_test_split

# Divida o dataset em features (X) e target (y)
X = df_encoded.drop(columns=['target'])
y = df_encoded['target']

# Divida em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

mlflow.set_experiment("telco_customer_churn")

with mlflow.start_run(run_name="telco_customer_churn"):
    model = LogisticRegression(random_state=42)
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    train_accuracy = accuracy_score(y_train, y_pred_train)
    test_accuracy = accuracy_score(y_test, y_pred_test)
    test_f1 = f1_score(y_test, y_pred_test)
    test_precision = precision_score(y_test, y_pred_test)
    test_recall = recall_score(y_test, y_pred_test)

    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("test_f1_score", test_f1)
    mlflow.log_metric("test_precision", test_precision)
    mlflow.log_metric("test_recall", test_recall)

    overfitting = train_accuracy - test_accuracy
    mlflow.log_metric("overfitting", overfitting)

    mlflow.sklearn.log_model(model, "model")

    print(f"=== LOGISTIC REGRESSION ===")
    print(f"Train Accuracy: {train_accuracy:.4f}")
    print(f"Test Accuracy:  {test_accuracy:.4f}")
    print(f"Test F1 Score:  {test_f1:.4f}")

    print(f"Test Precision: {test_precision:.4f}")
    print(f"Test Recall:    {test_recall:.4f}")
    print(f"Overfitting:    {overfitting:.4f}")

# Persistir o dataframe pré-processado

In [ ]:
df_encoded.to_csv("../data/Telco_customer_cuhrn_preprocessed.csv", index=False)

In [ ]:
# Persistir o modelo treinado com MLFlow

# Persistir o modelo usando joblib
import joblib
joblib.dump(model, "../models/baseline_model.joblib")